# Machine Learning Fundamentals: Classification
In the previous notebook, we focused on regression problems, where the goal is to predict a continuous target variable. In this notebook, we will shift our attention to **classification**, where the goal is to predict a categorical target variable (class).

Through a hands-on example, we will explore the entire workflow of a classification problem, from loading and visualizing the data to fitting and evaluating multiple classifiers. We will also discuss how to compare classifiers using metrics and confusion matrices.

## Learning objectives

By the end of this notebook, you will be able to:
- Load a classification dataset and identify features vs. labels (classes)
- Visualize the feature space
- Split data into train/test sets safely
- Fit and evaluate multiple classifiers
- Compare classifiers using metrics and confusion matrices

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("notebook")

print("Imports ready.")

## 1. Load and explore the dataset

We will use the [Phoneme dataset](https://www.openml.org/search?type=data&sort=runs&id=1489&status=active) for this exercise, which contains five harmonic features extracted from speech recordings. The harmonic features capture the spectral characteristics of the speech signal (i.e. the distribution of energy across different frequencies).

The task is to detect whether the recorded phoneme corresponds to an oral or nasal sound, based on the spectral features. Identifying phoneme types is a crucial first step in many speech processing applications, such as speech recognition and speaker identification. By learning to classify phonemes accurately, we can improve the performance of these systems and enable more natural interactions between humans and machines.

<div class="alert alert-block alert-info">
<b>QUESTIONS</b>

- How many samples and how many features do we have?
- Look at the feature values, do they need to be scaled? (Hint: look at the mean and standard deviation)
- Are the classes balanced? (Hint: use the describe function on the dataframe to look at the distribution of the target variable)
- Why can class imbalances make evaluation metrics like accuracy misleading?
</div>

In [ ]:
# Fetch dataset from OpenML (requires internet the first time; afterwards it may be cached)
phoneme = fetch_openml(name="phoneme", version=1, as_frame=True)

X = phoneme.data.copy().to_numpy()
y = phoneme.target.copy()

feature_names = phoneme.feature_names
target_labels = ['nasal' if label == '1' else 'oral' for label in y]
target_names = ['nasal', 'oral']

print(f"X shape: {X.shape} | y shape: {y.shape}")
print("Features:", feature_names)
print("Classes:", set(target_labels))

# Combine into one DataFrame for plotting
df = phoneme.frame.copy()
df['Class'] = target_labels
df.head()

***

Same as with regression, we will start by visualizing the feature space to get a better understanding of the data. This can help us to identify patterns, relationships, and potential challenges for classification. Here we will use a **pair plot** to helps us see how well classes separate in different feature combinations.

A pair plot (also called scatterplot matrix) is a grid of plots that shows pairwise combinations between multiple features in a dataset, all at once. In this grid, each row and column represents a feature. The off-diagonal cells show a scatter plot of one feature against another. These can help to find patterns like correlations, clusters or groupings, outliers, or nonlinear relationships between the features. The diagonal cells show the distribution of each feature (with a histogram or a KDE curve).

<div class="alert alert-block alert-info">
<b>QUESTIONS</b>

- Which relationships do you see between the features?
- Are the two classes clearly separated?
- Do you expect a linear classifier to work well here?
</div>

In [ ]:
sns.pairplot(
    df,
    hue="Class",
    corner=True,
    plot_kws={"alpha": 0.5, "s": 15}
)
plt.suptitle("Phoneme feature space")
plt.show()

***

Before we can fit a classification model, we need to prepare our data. This includes splitting the data into training and test sets, and optionally scaling the features.

**IMPORTANT:** add stratification to the train-test split by adding the parameter `stratify=y` in the split_train_test function.

<div class="alert alert-block alert-info">
<b>QUESTIONS</b>

- What could go wrong if we evaluate on the same data we train on?
- What does `stratify=y` do (look at the documentation from scikit-learn)?
</div>

In [ ]:
# Create a train-test split (80% train, 20% test), with stratification
X_train, X_test, y_train, y_test = _

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 2. Logistic Regression

Now that we have inspected and prepared our data, we can start fitting classification models!

We will start with Logistic Regression which is a strong baseline classifier. It is ideally suited for linearly separable datasets and can give surprisingly good performance even for complex classification problems. Logistic Regression is a popular model in classical machine learning because it is simple to train, has good interpretability, and gives a probabilistic output. The latter gives it a great advantage in domains where uncertainty quantification is important (e.g. in medical domains, where we might prefer the predictions to be expressed as a percentage instead of a binary yes-no answer).

The syntax for fitting a classifier in scikit-learn is very similar to that of regression models. We will create an instance of the `LogisticRegression` class, fit it to our training data, and then evaluate its performance on the test set.

<div class="alert alert-block alert-info">
<b>QUESTIONS</b>

- Look at the documenation of `LogisticRegression` in scikit-learn. What are the main parameters and what do they do?
- How do you explain the difference between the accuracy and balanced accuracy?
- If train accuracy is much higher than test accuracy, what might be happening?
</div>

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)

# Train the model on the (scaled) training data
log_reg.fit(X_train_scaled, y_train)

# Evaluate the trained model
y_train_pred = log_reg.predict(X_train_scaled)
y_test_pred = log_reg.predict(X_test_scaled)

print("Training set evaluation:")
acc = accuracy_score(y_train, y_train_pred)
bal_acc = balanced_accuracy_score(y_train, y_train_pred)
print(f"  Accuracy:          {acc:.4f}")
print(f"  Balanced accuracy: {bal_acc:.4f}")

print("\nTest set evaluation:")
acc = accuracy_score(y_test, y_test_pred)
bal_acc = balanced_accuracy_score(y_test, y_test_pred)
print(f"  Accuracy:          {acc:.4f}")
print(f"  Balanced accuracy: {bal_acc:.4f}")

***

#### Quick and easy evaluation metrics
Instead of computing all performance metrics separately, we can also use the [classification_report](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html) function from scikit-learn to get a quick overview of how well the model is doing. The classification report includes the precision, recall, f1-score, and support for each class (i.e. how many samples we have from each class), as well as the overall accuracy and macro/micro averages.

[Confusion matrices](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html) provide another quick insight into the predictions of the model. They are very popular in scientific reporting because they are visually intuitive and give a complete overview of all the correct and incorrect predictions.

<div class="alert alert-block alert-info">
<b>QUESTIONS</b>

- Do you see any patterns in the performance metrics?
- Is the model better at predicting one class and worse at the other?
</div>

In [ ]:
# Classification report
print("Classification report (Test):")
print(classification_report(y_test, y_test_pred, target_names=target_names))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred)

fig, ax = plt.subplots(figsize=(4, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(cmap=plt.cm.Blues, colorbar=False, ax=ax)
plt.grid(False)
plt.title("Confusion Matrix (Test set)")
plt.show()

## 3. Intermezzo - helper function to evaluate classifiers

To keep the notebook readable, we will define a helper function that:
- Fits the model
- Computes train/test accuracy and balanced accuracy
- Prints a classification report (precision/recall/F1)
- Plots a confusion matrix

We will use the same evaluation for all models so comparisons are fair.

<div class="alert alert-block alert-info">
<b>QUESTIONS</b>

- Exercise: complete the helper function by filling in the missing code (look at the code above for inspiration).
- Test your function by evaluating the logistic regression model again using this helper function (you should get the same results as before).
</div>

In [ ]:
def evaluate_classifier(model, X_tr, X_te, y_tr, y_te, title, class_names):
    """Fit a classifier, print metrics, plot confusion matrix, and return a summary row."""
    # Fit the model on the training data
    _

    # Make predictions on training and test data
    pred_tr = _
    pred_te = _

    # Compute the accuracy, balanced accuracy, and print classification report
    acc_tr = accuracy_score(y_tr, pred_tr)
    acc_te = accuracy_score(y_te, pred_te)
    bacc_tr = _
    bacc_te = _

    print(title)
    print(f"Train accuracy: {acc_tr:.3f} | Test accuracy: {acc_te:.3f}")
    print(f"Train balanced acc: {bacc_tr:.3f} | Test balanced acc: {bacc_te:.3f}")
    print()
    print("Classification report (Test):")
    print(classification_report(y_te, pred_te, target_names=class_names))

    # Plot the confusion matrix (test set)
    fig, ax = plt.subplots(figsize=(4, 4))
    cm = confusion_matrix(y_te, pred_te)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(cmap="Blues", values_format="d", colorbar=False, ax=ax)
    plt.grid(False)
    plt.title(f"Confusion matrix (Test) — {title}")
    plt.show()

    return {
        "model": title,
        "train_accuracy": acc_tr,
        "test_accuracy": acc_te,
        "train_balanced_accuracy": bacc_tr,
        "test_balanced_accuracy": bacc_te,
    }

# Test the helper function with logistic regression
res_lr = evaluate_classifier(
    log_reg,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test,
    title="Logistic Regression",
    class_names=target_names
)

## 3. Support Vector Machine (SVM)

Next we will train a Support Vector Machine classifier. SVMs are powerful classifiers that can model complex relationships between features and classes. By default, SVMs use a linear kernel, which means they will try to find a linear decision boundary. However, if the data is not linearly separable, we can use a non-linear kernel to allow the SVM to capture more complex relationships.

Because our dataset shows non-linear patterns, we will use an RBF kernel, which will allow the SVM to model non-linear relationships.

<div class="alert alert-block alert-info">
<b>QUESTIONS</b>

- Does SVM perform better or worse than Logistic Regression here?
- Why might a non-linear kernel help?
- Try to use a different kernel. How does it affect the performance? (Hint: look at the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html#sklearn.svm.SVC) to see which kernels are supported)
</div>

In [ ]:
# Initialize an SVM with RBF kernel
svm = SVC(kernel="rbf", random_state=42)

# Evaluate the classifier
res_svm = evaluate_classifier(
    svm,
    X_train_scaled, X_test_scaled,
    y_train, y_test,
    title="SVM (RBF kernel, scaled features)",
    class_names=target_names
)

## 4. Decision Tree

Decision trees can learn non-linear decision boundaries and do **not** require scaling. They also have a very intuitive prediction pattern with a series of sequential splitting decisions. This makes decision trees ideal for cases where you want to have clear interpretability and traceability of why certain predictions were made. For more complex classification problems and large numbers of features the decision trees can however quickly become too large and lose their interpretability.

<div class="alert alert-block alert-info">
<b>QUESTIONS</b>

- Why do decision trees not require feature scaling?
- What might happen if we let a tree grow without limits?
- Look at the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) of the DecisionTreeClassifier. What are the main parameters and what do they do?
</div>

In [ ]:
# Initialize a Decision Tree classifier (choose your own hyperparameters if you like)
tree = _

# Evaluate the classifier
res_tree = evaluate_classifier(
    tree,
    X_train, X_test,
    y_train, y_test,
    title="Decision Tree",
    class_names=target_names
)

***

One of the main advantages of decision trees is that they are very interpretable. Let's take a look at the structure of our trained decision tree.

We can use the `plot_tree` function from scikit-learn to visualize the structure of the decision tree. This will show us how the tree splits the data based on different features and thresholds, and how it makes predictions at each leaf node.

<div class="alert alert-block alert-info">
<b>QUESTIONS</b>

- Does this provide a clear and intuitive overview of the decision tree model?
- How could we improve the interpretability of the model?
</div>

In [ ]:
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(12, 8))
plot_tree(
    tree,
    feature_names=feature_names,
    class_names=target_names,
    filled=True,
    rounded=True,
    fontsize=8,
    ax=ax
)
plt.title("Decision Tree Structure")
plt.show()

## 5. Random Forest

Ensemble methods combine the predictions of multiple models to improve performance and reduce overfitting. A popular example of an ensemble method is the Random Forest, which consists of a collection of decision trees. Each tree is trained on a random subset of the data and a random subset of the features, which helps to reduce overfitting and improve generalization. In the end the predictions of all the trees are combined (e.g. by majority voting) to make the final prediction.

<div class="alert alert-block alert-info">
<b>QUESTIONS</b>

- Why might random forest generalize better than a single decision tree?
- Do you expect training accuracy to be very high for a random forest?
</div>

In [ ]:
forest = RandomForestClassifier(n_estimators=200, random_state=42)

res_forest = evaluate_classifier(
    forest,
    X_train, X_test,
    y_train, y_test,
    title="Random Forest",
    class_names=target_names
)

## 6. Compare model performance

Now that we have trained and evaluated multiple classifiers, we can compare their performance to see which one works best for our dataset.

We will put the key metrics into a table to compare models quickly. This will allow us to see at a glance which model has the highest accuracy, balanced accuracy, precision, recall, and F1-score.

> Tip: If models look very similar, use the confusion matrices to understand *which mistakes* are made.

<div class="alert alert-block alert-info">

<b>QUESTIONS</b>
- Which model performs best on this dataset?
- Do you see any patterns in the performance metrics across models?
</div>

In [ ]:
results = pd.DataFrame([res_lr, res_svm, res_tree, res_forest]).set_index("model")
results.sort_values("test_accuracy", ascending=False)

## Wrap-up

In this notebook we learned how to approach a classification problem from start to finish. We:
- Visualized class separation with a pair plot
- Trained and evaluated four common classifiers
- Compared results using metrics and confusion matrices

The pattern for training and evaluating classifiers is very similar to that of regression models. We can use the same workflow and evaluation strategy for different types of models, which makes it easy to compare their performance. As you can see from what we have done here, training a machine learning model is not the hard part. The hard part is to understand the data, prepare it well, choose the right model, and interpret the results. This is where domain knowledge and experience come into play.

In the next session we will learn how to tune and compare models more systematically (cross-validation, pipelines, and hyperparameter search).

---

## Optional exercises (for early finishers)

### A. Inspect misclassified samples

Goal: identify which test samples were misclassified and look for patterns.

<div class="alert alert-block alert-info">
<b>QUESTIONS</b>

- Which classes are most often confused?
- Are misclassified samples near overlaps you saw in the pair plot?
</div>

In [ ]:
# Choose a model to inspect (try: log_reg, svm, tree, forest)
best_model = forest

pred = best_model.predict(X_test_scaled)
wrong_idx = np.where(pred != y_test)[0]
print("Number of misclassified test samples:", len(wrong_idx))

In [ ]:
feature_0 = 0
feature_1 = 1

fig, ax = plt.subplots(figsize=(6, 6))

# Plot all test points with transparency
ax.scatter(
    X_test[:, feature_0],
    X_test[:, feature_1],
    c=['blue' if label == 0 else 'red' for label in y_test],
    alpha=0.2,
    s=10,
    label="Test samples"
)

# Add a scatter plot of the missclassified points
# Hint: use facecolor='none' to get unfilled markers

# -------------------------------
# INSERT YOUR OWN CODE HERE
# -------------------------------

ax.set_xlabel(feature_names[feature_0])
ax.set_ylabel(feature_names[feature_1])
ax.legend(loc='upper right', fontsize='small')
plt.title("Misclassified Test Samples Highlighted")
plt.show()

### B. Manual hyperparameter sensitivity

Each classifier has a set of hyperparameters that can be tuned and will have impact on the model's performance. Choose one classifier and try changing one parameter manually and re-run evaluation.

Examples:
- SVM: `C` (e.g., 0.1, 1, 10)
- Decision Tree: `max_depth` (e.g., 1, 2, 3, None)
- Random Forest: `n_estimators` or `max_depth`

<div class="alert alert-block alert-info">
<b>QUESTIONS</b>

- Which parameter changes reduce overfitting?
- Do results change a lot, or are they stable?
</div>

In [ ]:
# Example: a shallower tree can reduce overfitting
tree_small = DecisionTreeClassifier(max_depth=3, random_state=42)

_ = evaluate_classifier(
    tree_small,
    X_train, X_test,
    y_train, y_test,
    title="Decision Tree (max_depth=3)",
    class_names=target_names
)

### C. Model evaluation: ROC curve
Next to the classification report and confusion matrix, the Receiver Operating Characteristing (ROC) curve is another popular way of visualizing the performance of a classifier. All of the evaluation metrics that we have seen before depend on the threshold that is used during the evaluation process (i.e. is a sample classified as positive if the prediction is higher than 0.5, or 0.6, or, ...?). The ROC curve bypasses this issue by computing the True Positive Rate and False Positive Rate at a range of possible thresholds, and plotting the values on one curve.

A completely random classifier will have a ROC curve that follows the diagonal line (i.e. TPR = FPR), while a perfect classifier will have a curve that goes through the top-left corner (i.e. TPR = 1, FPR = 0). The ROC curve also has an associated performance metric called AUC (appropriately named so because it is defined by calculating the Area Under the Curve). The AUC is popular for comparing different models, because as we mentioned before, it is independent of the threshold and can handle class imbalances relatively well.

Here we will plot the ROC curve for our trained models.

<div class="alert alert-block alert-info">
<b>QUESTIONS</b>

- How do you interpret the differences between these curves?
- If I want to deploy the Random Forest model with a specificity of 90%, what will the sensitivity be? What about the SVM model? (Hint: False Positive Rate = 1 - specificity; True Positive Rate = sensitivity)
</div>

In [ ]:
from sklearn.metrics import RocCurveDisplay

# Define classifiers to evaluate
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "SVM (RBF kernel)": SVC(kernel="rbf", gamma="scale", C=1.0, probability=True,random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42)
}

# Plot ROC curves
fig, ax = plt.subplots(figsize=(6, 6))
for name, model in classifiers.items():
    model.fit(X_train_scaled, y_train)
    y_test_pred = model.predict_proba(X_test_scaled)
    
    RocCurveDisplay.from_predictions(y_test, y_test_pred[:, 1], pos_label='2', name=name, ax=ax)
plt.title("ROC Curve")
plt.show()

### D. Feature importance

Some classifiers (e.g. decision trees and random forests) can provide insights into which features are most important for making predictions. This can help us to understand the underlying patterns in the data and to identify which features are most relevant for the classification task. In the case of random forests, the feature importance is typically calculated based on how much each feature contributes to reducing the impurity (e.g. Gini impurity or entropy) across all the trees in the forest. The more a feature is used to make important splits in the trees, the higher its importance score will be.

You can access the feature importance scores from the `feature_importances_` attribute of the trained random forest model.

<div class="alert alert-block alert-info">
<b>QUESTIONS</b>

- Which features seem most important?
- Does this match with what you saw in the pair plot?
</div>

In [ ]:
importances = pd.Series(forest.feature_importances_, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(6, 3))
importances[::-1].plot(kind="barh")
plt.title("Random Forest feature importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

***

Now do the same for your trained Logistic Regression model. Look at the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) to see how you can extract the coefficients from the LogisticRegression model.

<div class="alert alert-block alert-info">
<b>QUESTIONS</b>

- Are these the same features that were important for Random Forest?
- Do you expect the feature importance to be the same for every model?
</div>

In [ ]:
# -------------------------------
# INSERT YOUR OWN CODE HERE
# -------------------------------